# Week 1.4: Prompt Engineering for Digital Twins

## Learning Objectives
- Master different prompting techniques
- Understand zero-shot, few-shot, and chain-of-thought
- Apply prompting to digital twin interactions

## Key Insight
> 'Moving from Chat to System Instructions'

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict

print('✅ Ready for prompting exercises!')

✅ Ready for prompting exercises!


## Part 1: Prompting Fundamentals

### 1.1 Zero-Shot Prompting

In [2]:
# Example prompts for digital twin
zero_shot_examples = [
    {
        'task': 'activity_prediction',
        'prompt': 'Predict the next activity: I just finished work, had dinner, and now...',
        'expected': 'leisure/relaxation activity'
    },
    {
        'task': 'preference_extraction',
        'prompt': 'Extract preferences from: I love outdoor activities but hate crowded places',
        'expected': {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}
    }
]

print('Zero-Shot Examples:')
for ex in zero_shot_examples:
    print(f"\nTask: {ex['task']}")
    print(f"Prompt: {ex['prompt']}")
    print(f"Expected: {ex['expected']}")

Zero-Shot Examples:

Task: activity_prediction
Prompt: Predict the next activity: I just finished work, had dinner, and now...
Expected: leisure/relaxation activity

Task: preference_extraction
Prompt: Extract preferences from: I love outdoor activities but hate crowded places
Expected: {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}


### 🎯 Exercise 1.1 (Easy): Create Your Prompts

**Task**: Design prompts for your digital twin use case.

In [5]:
# YOUR CODE HERE
!pip install -q groq

import os
from groq import Groq
import os, json
#IMPORTANT: Set your real API key securely.
#Option 1 (recommended): Set this in your environment before running the notebook:
#   %env GROQ_API_KEY=gsk_.....

#Option 2 (Only for quick local testing): Paste it here (NOT in shared notebooks)
os.environ["GROQ_API_KEY"] = "gsk...."

client=Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

def llm(prompt: str) -> str:
  """Simple helper for single-turn prompts."""
  chat_completion = client.chat.completions.create(
      model="openai/gpt-oss-120b",
      messages=[
          {"role": "user", "content": prompt}
      ]
  )
  return chat_completion.choices[0].message.content

print("Connected. Ready.")

Connected. Ready.


In [6]:
# Zero-shot Prompting
#####################################

# Zero-shot: describe the task only
# No examples, no format instructions
# Model infers the answer from training

prompt = '''
I just finished AI studies, I'm tired
and now
'''
response = llm(prompt)
print(response)

Congratulations on wrapping up your AI studies—that’s a huge milestone! 🎉 It’s completely natural to feel drained after such an intense stretch of learning and problem‑solving. Here are a few ideas to help you recharge and transition smoothly into the next phase:

---

## 1️⃣ Give Your Brain a Real Break  
- **Power‑down:** Put your devices on “Do Not Disturb” for at least an hour. Even a short digital detox can reset your focus.  
- **Micro‑naps:** A 10–20 minute nap can boost alertness without leaving you groggy.  
- **Nature reset:** A walk outside (even around the block) gets fresh oxygen and a dose of natural light—both proven to improve mood and cognition.

## 2️⃣ Celebrate the Win  
- **Treat yourself:** Whether it’s a favorite coffee, a slice of cake, or a new book, make a small, tangible reward.  
- **Share the news:** Let friends or family know—talking about your accomplishment reinforces its significance.  
- **Mini‑ceremony:** Write a short “graduation” note to yourself hig

In [17]:
# Few-shot Prompting
###################################

# Few-shot: show examples, do not explain
# 2-3 input/output pairs before the real question
# Model learns format from demonstration

prompt = '''
Examples:
Input: Finished Running, shower, coffee
Output: {"next":"Study","confidence":0.82}

Input: Watched TV, dinner
Output: {"next":"Sleep":"confidence":0.91}

Input: Finished exercising, had shower, now
Output: '''

response = llm(prompt)
print(response)


{"next":"Eat","confidence":0.87}


In [18]:
# Chain-Of-Thought Prompting vs Plain
###################################

# Chain-Of-Thought: four extra words.
# Append: Think step by step.
# Model shifts to explicit reasoning.

plain = '''Should I exercise today?
Context: worked 10h, slept 5h.'''

cot = '''Should I exercise today?
Context: worked 10h, slept 5h.
Think step by step.'''

print('--- PLAIN ---')
print(llm(plain))
print()
print('--- CHAIN-OF-THOUGHT ---')
print(llm(cot))

--- PLAIN ---
**Short answer:** Yes – but keep it light and listen to your body.

---

### Why a little movement can still be a good idea

| Benefit of a light‑to‑moderate session | Why it matters after a 10‑hour workday & 5‑hour night |
|----------------------------------------|-------------------------------------------------------|
| **Improves mood & reduces stress**     | Physical activity releases endorphins that can counteract the mental fatigue that often follows a long shift. |
| **Boosts alertness**                   | A brief walk or gentle stretch can increase blood flow to the brain and help shake off that “groggy” feeling. |
| **Supports sleep quality**             | Light exercise (especially earlier in the day) can make it easier to fall asleep and achieve deeper REM cycles the next night. |
| **Protects against stiffness**         | Sitting for many hours compresses muscles and joints; a short mobility routine can keep you from feeling “locked up.” |

---

### How to e

In [22]:
# 2.2 - Build The Twin's System Introduction
####################################################

system = '''
ROLE: You are a digital twin for {user}.
You represent their preferences and
behavioural patterns.

KNOWLEDGE: You have the user's activity
history and temporal patterns.

CONSTRAINTS: Always return structured
output. Never guess. If data is missing,
say so explicitly.

TONE: Precise, analytical.
'''

r = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role":"system","content":system},
        {"role":"user","content":"What next?"}
    ]
)
print(r.choices[0].message.content)

{
  "status": "insufficient_data",
  "message": "I cannot determine the appropriate next step without additional context.",
  "missing_information": [
    "current_task_or_activity",
    "desired_goal_or_outcome",
    "time_of_day_or_schedule_constraints",
    "any specific preferences relevant to the next action"
  ],
  "next_action_suggestion": "Please provide the above details so I can generate a precise recommendation."
}


In [26]:
# 2.2 - Build the Twin's System Instructions
#################################################

user = "Omar"

user_profile = {
    "name": "Omar",
    "role": "Univeristy lecturer",
    "preferences": [
        "prefers analytical explanations",
        "likes structured responses",
        "works often with Python and presentations"
    ],
    "goals": [
        "improve teaching materials",
        "prepare technical lectures",
        "build research workflows"
    ]
}

activity_history = [
    {"time": "2026-07-10", "activity": "Created slides on edge computing"},
    {"time": "2026-07-12", "activity": "Worked onPython code examples for class"},
    {"time": "2026-07-14", "activity": "Prepared assessment questions on digital systems"}
]

temporal_patterns = {
    "most_active_period": "evenings",
    "common_tasks": [
        "lecture preperation",
        "technical writing",
        "coding examples"
    ],
    "recurring_focus": [
        "emerging technologies",
        "computer engineering education"
    ]
}

system = f"""
ROLE:
You are a digital twin for {user}.
You simulate the user's likely priorities, preferences, and next actions
using only the evidence provided.

AVAILABLE DATA:
1. User profile:
{user_profile}

2. Activity history:
{activity_history}

3. Temporal patterns:
{temporal_patterns}

OBJECTIVE:
Given a user query, produce an informed and structured assessment of:
- the user's likely current context,
- the most probable next actions,
- the evidence supporting each action,
- any missing data that limits confidence.

REASONING RULES:
- Use only the supplied data.
- Do not invent facts.
- Separate observed facts from inferred conclusions.
- If evidence is weak or missing, say so explicitly.
- Rank likely next actions by confidence.
- Ground every inference in profile, history, or temporal patterns.

OUTPUT FORMAT:
Return valid JSON with this structure:
{{
  "user": "string",
  "current_context": {{
    "summary": "string",
    "observed_signals": ["string"]
  }},
  "predicted_next_actions": [
    {{
      "action": "string",
      "confidence": "high | medium | low",
      "why": ["string"],
      "supporting_data": ["string"]
    }}
  ],
  "missing_information": ["string"],
  "response_quality": {{
    "grounded_in_data": true,
    "contains_guessing": false
  }}
}}

TONE:
Precise, analytical, concise.
"""

In [27]:
r = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role":"system","content":system},
        {"role":"user","content":"What next?"}
    ]
   # response_format={ "type": "json_object" }  #use only if supported by your client/model
)

print(r.choices[0].message.content)

{
  "user": "Omar",
  "current_context": {
    "summary": "Evening work session focused on preparing teaching materials for an emerging‑technology lecture (edge computing) with recent activities on slides, Python examples, and assessment questions.",
    "observed_signals": [
      "Recent creation of slides on edge computing (2026-07-10)",
      "Recent development of Python code examples for class (2026-07-12)",
      "Recent preparation of assessment questions on digital systems (2026-07-14)",
      "Temporal pattern: most active in evenings",
      "Profile preference for analytical, structured responses and work with Python and presentations",
      "Recurring focus on emerging technologies and computer engineering education"
    ]
  },
  "predicted_next_actions": [
    {
      "action": "Finalize and polish the edge‑computing lecture slides for upcoming class",
      "confidence": "high",
      "why": [
        "Slides were created earliest in the sequence, indicating they are a 